# Experimento de 140 combinaciones, parte 4: comparación de optimizadores

La guía pide demostrar con números si la búsqueda bayesiana y la genética son más eficientes que la grilla y la aleatoria. Con el mismo
presupuesto (30 evaluaciones por búsqueda), los mismos inner folds y un solo hilo por modelo, se comparan en cinco ejes:

1. la métrica externa a la que llegan;
2. el tiempo total y por evaluación;
3. qué tan rápido llegan a su mejor valor (curvas de desempeño en cualquier momento);
4. la calidad por unidad de presupuesto, con una prueba pareada entre combinaciones;
5. la estabilidad ante la semilla.

Al final, el halving sucesivo (multi-fidelidad), corrido aparte y sin balanceo.

## Pipeline

![Pipeline del experimento](fig_pipeline_experimento.png)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as st
from src import analisis, config, runner, stats

OPT = ["grilla", "aleatoria", "bayesiana", "genetica"]
COLOR = {"grilla": "#7F7F7F", "aleatoria": "#1A1A1A", "bayesiana": "#C0141F", "genetica": "#E8898F", "halving": "#4C72B0"}
maestra, res = analisis.cargar()
trazas = pd.read_parquet(config.TRAZAS)
trazas = trazas[trazas.optimizador.isin(OPT)]
if "fraccion" not in trazas:
    trazas["fraccion"] = 1.0
trazas["fraccion"] = trazas["fraccion"].fillna(1.0)
combos = res[["tarea", "modelo", "balanceo"]].drop_duplicates()
completas = res.groupby(["tarea", "modelo", "balanceo"]).optimizador.nunique()
completas = completas[completas == 4].index
res4 = res.set_index(["tarea", "modelo", "balanceo"]).loc[completas].reset_index()
print(f"Grupos (tarea, modelo, balanceo) con los 4 optimizadores completos: {len(completas)}")

## 1. Métrica externa por optimizador

Para cada grupo (tarea, modelo, balanceo) se ordenan los cuatro optimizadores por su métrica externa (F1 macro en clasificación, R² en
regresión). Rango 1 es el mejor.

In [ ]:
res4["externa"] = np.where(res4.tarea == "clf", res4.f1_macro, res4.r2)
res4["rango"] = res4.groupby(["tarea", "modelo", "balanceo"]).externa.rank(ascending=False)
ganador = res4.loc[res4.groupby(["tarea", "modelo", "balanceo"]).externa.idxmax()]
tabla = pd.DataFrame({
    "rango medio": res4.groupby("optimizador").rango.mean(),
    "veces mejor": ganador.optimizador.value_counts(),
    "métrica externa media": res4.groupby("optimizador").externa.mean(),
}).reindex(OPT).fillna(0)
piv_ext = res4.pivot_table(index=["tarea", "modelo", "balanceo"], columns="optimizador", values="externa")[OPT]
tabla["diferencia media con la grilla"] = piv_ext.sub(piv_ext["grilla"], axis=0).mean()
tabla.round(4)

In [ ]:
piv = res4.pivot_table(index=["tarea", "modelo", "balanceo"], columns="optimizador", values="externa")[OPT]
filas = []
for a in OPT:
    for c in OPT:
        if a < c:
            w = st.wilcoxon(piv[a], piv[c], zero_method="zsplit")
            filas.append({"comparación": f"{a} vs {c}", "mediana de la diferencia": float(np.median(piv[a] - piv[c])),
                          "p Wilcoxon": w.pvalue, "delta de Cliff": stats.cliff_delta(piv[a], piv[c])["delta"]})
w = pd.DataFrame(filas)
w["p Holm"] = stats.holm(w["p Wilcoxon"])
w.set_index("comparación").round(4)

Prueba de rangos con signo de Wilcoxon, pareada por grupo (Wilcoxon, 1945), con p ajustado por Holm. La delta de Cliff dice qué tan seguido un
optimizador queda por encima del otro, sin depender de la escala de la métrica.

## 2. Tiempo y evaluaciones

El tiempo por evaluación depende sobre todo del modelo y de los hiperparámetros que el optimizador decide probar (más árboles, más vecinos,
C más grande), no del optimizador en sí. El costo propio de TPE y del genético es despreciable frente a entrenar un modelo.

In [ ]:
t = res4.groupby("optimizador")[["evaluaciones", "t_busqueda", "t_por_evaluacion"]].agg(["mean", "sum"])
t.columns = [f"{a} ({'media' if b == 'mean' else 'total'})" for a, b in t.columns]
t = t[["evaluaciones (media)", "t_busqueda (media)", "t_busqueda (total)", "t_por_evaluacion (media)"]]
t["horas-núcleo de búsqueda"] = t["t_busqueda (total)"] * config.K_EXTERNO / 3600 / config.K_EXTERNO
t.reindex(OPT).round(2)

In [ ]:
por_modelo = res4.pivot_table(index=["tarea", "modelo"], columns="optimizador", values="t_busqueda", aggfunc="mean")[OPT]
por_modelo.round(1)

## 3. Curvas de desempeño en cualquier momento

Para cada búsqueda (grupo y pliegue externo) se toma el mejor puntaje interno visto hasta la evaluación t. Para poder promediar entre modelos
con escalas distintas, se normaliza como *regret*: 0 es el mejor valor que encontró cualquiera de los cuatro optimizadores en esa búsqueda y 1
es la mediana de todas sus evaluaciones; se acota en 1 (un mejor valor por debajo de la mediana cuenta como 1, para que un primer candidato desastroso no domine el promedio). Menor es mejor.

La grilla evalúa en orden lexicográfico, así que su curva depende de ese orden: es la curva de quien la recorre de principio a fin.

In [ ]:
tz = trazas[trazas.fraccion == 1.0].copy()
tz = tz[np.isfinite(tz.puntaje)]
claves = ["tarea", "modelo", "balanceo", "pliegue"]
tz = tz.merge(pd.DataFrame(list(completas), columns=["tarea", "modelo", "balanceo"]), on=["tarea", "modelo", "balanceo"])
tz = tz.sort_values(claves + ["optimizador", "evaluacion"])
tz["mejor_hasta"] = tz.groupby(claves + ["optimizador"]).puntaje.cummax()
tz["t_acumulado"] = tz.groupby(claves + ["optimizador"]).segundos.cumsum()
ref = tz.groupby(claves).puntaje.agg(maximo="max", mediana="median")
tz = tz.join(ref, on=claves)
tz["regret"] = ((tz.maximo - tz.mejor_hasta) / (tz.maximo - tz.mediana).replace(0, np.nan)).clip(0, 1)

def curva(df):
    # cada búsqueda se extiende hasta 30 con su último valor, para que la grilla (24 o 27 evaluaciones) no salga de la cuenta
    filas = []
    for _, g in df.groupby(claves + ["optimizador"]):
        r = g.set_index("evaluacion").regret.reindex(range(1, config.PRESUPUESTO + 1)).ffill()
        filas.append(r.rename(g.optimizador.iloc[0]))
    return pd.concat(filas, axis=1).T.groupby(level=0).mean().T

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, tarea, titulo in zip(axes, ["clf", "reg"], ["Clasificación", "Regresión"]):
    sub = tz[tz.tarea == tarea]
    if sub.empty:
        ax.axis("off"); continue
    c = curva(sub)
    for o in OPT:
        if o in c:
            ax.plot(c.index, c[o], label=o, color=COLOR[o], lw=2)
    ax.set_title(f"{titulo}: regret medio según el número de evaluaciones"); ax.set_xlabel("evaluación"); ax.set_ylabel("regret normalizado")
    ax.set_yscale("symlog", linthresh=0.05); ax.legend()
fig.tight_layout(); fig.savefig("fig_exp_anytime.png", dpi=130, bbox_inches="tight"); plt.show()

## 4. Calidad por unidad de presupuesto

Tres medidas por búsqueda, promediadas:

- **Área bajo la curva de regret** (de la evaluación 1 a la 30, dividida por 30): resume toda la curva; menor es mejor.
- **Evaluaciones hasta el 95 %**: cuántas evaluaciones tarda en cerrar el 95 % de la distancia entre la mediana y el mejor valor de esa búsqueda.
- **Mejor valor final** (regret en la evaluación 30).

La prueba pareada (Wilcoxon por búsqueda, con Holm) dice si las diferencias entre optimizadores son sistemáticas o ruido.

In [ ]:
filas = []
for k, g in tz.groupby(claves + ["optimizador"]):
    r = g.set_index("evaluacion").regret.reindex(range(1, config.PRESUPUESTO + 1)).ffill().bfill()
    llega = r[r <= 0.05]
    filas.append({**dict(zip(claves + ["optimizador"], k)), "area_regret": r.mean(), "regret_final": r.iloc[-1],
                  "evals_hasta_95": llega.index.min() if len(llega) else np.nan, "segundos": g.segundos.sum()})
ef = pd.DataFrame(filas)
display(ef.groupby("optimizador")[["area_regret", "regret_final", "evals_hasta_95", "segundos"]].mean().reindex(OPT).round(3))

piv = ef.pivot_table(index=claves, columns="optimizador", values="area_regret")[OPT].dropna()
filas = []
for a, c in [("bayesiana", "grilla"), ("bayesiana", "aleatoria"), ("genetica", "grilla"), ("genetica", "aleatoria"),
             ("aleatoria", "grilla"), ("bayesiana", "genetica")]:
    w = st.wilcoxon(piv[a], piv[c], zero_method="zsplit")
    filas.append({"comparación": f"{a} vs {c}", "búsquedas": len(piv), "el primero mejor (%)": 100 * np.mean(piv[a] < piv[c]),
                  "mediana de la diferencia de área": float(np.median(piv[a] - piv[c])), "p": w.pvalue})
ef_t = pd.DataFrame(filas)
ef_t["p Holm"] = stats.holm(ef_t.p)
ef_t.set_index("comparación").round(4)

## 5. Diversidad del algoritmo genético

Si la diversidad cae a cero muy pronto, la población convergió prematuramente y el genético deja de explorar. Distancia media entre individuos en
el cubo unitario (0 = todos iguales) e individuos distintos de una población de 6.

In [ ]:
ruta = config.RUNS / "diversidad.parquet"
if ruta.exists():
    dv = pd.read_parquet(ruta)
    dv = dv[dv.optimizador == "genetica"]
    g = dv.groupby(["tarea", "generacion"])[["distancia_media", "distintos"]].mean().unstack("tarea")
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
    g["distancia_media"].plot(ax=axes[0], marker="o", color=["#C0141F", "#1A1A1A"][:g["distancia_media"].shape[1]])
    axes[0].set_title("Distancia media entre individuos"); axes[0].set_xlabel("generación")
    g["distintos"].plot(ax=axes[1], marker="o", color=["#C0141F", "#1A1A1A"][:g["distintos"].shape[1]])
    axes[1].set_title("Individuos distintos (de 6)"); axes[1].set_xlabel("generación")
    fig.tight_layout(); fig.savefig("fig_exp_diversidad.png", dpi=130, bbox_inches="tight"); plt.show()
    display(dv.groupby("modelo")[["distancia_media"]].agg(["first", "last"]).round(3))

## 6. Halving sucesivo (multi-fidelidad)

Corrido sin balanceo para los 14 modelos. Mira 90 candidatos con 1/9 de los datos, 30 con 1/3 y 10 con todos, por el mismo costo que 30
evaluaciones completas. Se compara contra los otros cuatro en las mismas combinaciones.

In [ ]:
m_all, r_all = analisis.cargar(incluir_halving=True)
hv = r_all[(r_all.balanceo.isin(["ninguno", runner.SIN_BALANCEO]))]
if "halving" in set(hv.optimizador):
    hv = hv.assign(externa=np.where(hv.tarea == "clf", hv.f1_macro, hv.r2))
    comp = hv.pivot_table(index=["tarea", "modelo"], columns="optimizador", values="externa")
    display(comp[[c for c in OPT + ["halving"] if c in comp]].round(4))
    tiempo = hv.pivot_table(index=["tarea", "modelo"], columns="optimizador", values="t_busqueda")
    display(tiempo[[c for c in OPT + ["halving"] if c in tiempo]].round(1))
else:
    print("El halving todavía no termina.")

## 7. Estabilidad ante la semilla

La semilla cambia los pliegues, los candidatos aleatorios, el muestreador de TPE, la población inicial del genético y la aleatoriedad de los
modelos. Se repite un subconjunto (Random Forest, XGBoost y SVM, sin balanceo, con los cuatro optimizadores) con otras semillas y se mira
cuánto cambia la métrica externa.

In [ ]:
partes = runner.tabla_partes()
partes = partes[partes.optimizador.isin(OPT)]
sem = partes.groupby(["tarea", "modelo", "balanceo", "optimizador", "semilla"]).agg(
    externa=("f1_macro", "mean"), r2=("r2", "mean") if "r2" in partes else ("f1_macro", "mean"), pliegues=("pliegue", "nunique")).reset_index()
sem = sem[sem.pliegues == config.K_EXTERNO]
sem["externa"] = np.where(sem.tarea == "clf", sem.externa, sem.r2)
varias = sem.groupby(["tarea", "modelo", "balanceo", "optimizador"]).semilla.nunique()
varias = varias[varias > 1].index
if len(varias):
    s = sem.set_index(["tarea", "modelo", "balanceo", "optimizador"]).loc[varias].reset_index()
    out = s.groupby(["tarea", "modelo", "optimizador"]).externa.agg(["mean", "std", "min", "max", "count"])
    display(out.round(4))
else:
    print("Las corridas con otras semillas todavía no terminan.")

## 8. Interpretación

(se completa con la corrida completa)

## Referencias

- Akiba, T., Sano, S., Yanase, T., Ohta, T. y Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. En *Proceedings of the 25th ACM SIGKDD* (pp. 2623-2631). https://doi.org/10.1145/3292500.3330701
- Bergstra, J., Bardenet, R., Bengio, Y. y Kégl, B. (2011). Algorithms for hyper-parameter optimization. *Advances in Neural Information Processing Systems, 24*.
- Bergstra, J. y Bengio, Y. (2012). Random search for hyper-parameter optimization. *Journal of Machine Learning Research, 13*, 281-305.
- Cliff, N. (1993). Dominance statistics: Ordinal analyses to answer ordinal questions. *Psychological Bulletin, 114*(3), 494-509. https://doi.org/10.1037/0033-2909.114.3.494
- Fortin, F.-A., De Rainville, F.-M., Gardner, M.-A., Parizeau, M. y Gagné, C. (2012). DEAP: Evolutionary algorithms made easy. *Journal of Machine Learning Research, 13*, 2171-2175.
- Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics, 6*(2), 65-70.
- Li, L., Jamieson, K., DeSalvo, G., Rostamizadeh, A. y Talwalkar, A. (2018). Hyperband: A novel bandit-based approach to hyperparameter optimization. *Journal of Machine Learning Research, 18*(185), 1-52.
- Wilcoxon, F. (1945). Individual comparisons by ranking methods. *Biometrics Bulletin, 1*(6), 80-83. https://doi.org/10.2307/3001968